In [1]:
import json
import os
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseTextEmbedding
from openai import OpenAI
from deepeval import evaluate
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRecallMetric,
    ContextualPrecisionMetric
)
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM
from dotenv import load_dotenv

c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

In [3]:
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

Model for generting final answer

In [4]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    timeout=120.0
)

In [5]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3368.86it/s]


In [6]:
def search_with_reranker(query: str, model: SentenceTransformer, collection: str, initial_k: int = 10, final_k: int = 3):
    print(f"Запит: '{query}'")
    
    original_query_text = query 
    
    query_vector = model.encode("query: " + query)

    search_results = client.query_points(
        collection_name=collection,
        prefetch=[
            models.Prefetch(
                query=query_vector,
                using="default",
                limit=initial_k
            )
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=initial_k
    )

    if not search_results or not search_results.points:
        return "На жаль, за цим запитом нічого не знайдено."

    documents = []
    for hit in search_results.points:
        text = hit.payload.get("text", "")
        source = hit.payload.get("file_name", "Невідомий документ")
        documents.append({"text": text, "source": source})

    cross_input = [[original_query_text, doc["text"]] for doc in documents]
    
    rerank_scores = reranker.predict(cross_input)

    for i in range(len(documents)):
        documents[i]["cross_score"] = float(rerank_scores[i])

    documents = sorted(documents, key=lambda x: x["cross_score"], reverse=True)

    context_blocks = []
    for doc in documents[:final_k]:
        block = f"[Джерело: {doc['source']} | Реранк: {doc['cross_score']:.2f}]\n{doc['text']}"
        context_blocks.append(block)

    return "\n\n---\n\n".join(context_blocks)

In [7]:
def generate_final_answer(query: str, retrieved_context: str, answer_model: str):
    system_prompt = """
    Ти є інтелектуальним помічником для працівників Українського Католицького Університету (УКУ).
    Твоє завдання — відповісти на запитання користувача, спираючись ВИКЛЮЧНО на наданий контекст з внутрішніх документів.

    ПРАВИЛА:
    1. Якщо в контексті немає відповіді, чесно скажи: "На жаль, у знайдених документах немає інформації для відповіді на це запитання." Не вигадуй жодних фактів.
    2. Формулюй відповідь чітко, професійно та структуровано.
    3. Обов'язково вказуй джерело інформації у форматі: [Джерело: Назва_файлу.pdf].
    """

    user_message = f"КОНТЕКСТ З БАЗИ ДАНИХ:\n{retrieved_context}\n\nЗАПИТАННЯ КОРИСТУВАЧА:\n{query}"

    response = llm_client.chat.completions.create(
        model=answer_model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.1,
            max_tokens=1000,
            timeout=120.0
    )
    return response.choices[0].message.content

In [8]:
with open("final_golden_dataset.json", "r", encoding="utf-8") as f:
    golden_data = json.load(f)

In [9]:
def rag_pipeline(query, model, collection, answer_model):
    result = search_with_reranker(query, model, collection)
    answer = generate_final_answer(query, result, answer_model)
    return result, answer

In [10]:
def create_test_cases(model, collection, answer_model):
    test_cases = []

    for item in golden_data:
        
        contexts, prediction = rag_pipeline(item["input"], model, collection, answer_model)
        if isinstance(contexts, str):
            contexts = [contexts]
        contexts = [str(c) for c in contexts]

        test_case = LLMTestCase(
            input=item["input"],
            actual_output=prediction,
            expected_output=item["expected_output"],
            retrieval_context=contexts
        )

        test_cases.append(test_case)

    return test_cases

In [11]:
class OpenRouterLLM(DeepEvalBaseLLM):

    def __init__(self):
        self.client = OpenAI(
            api_key=OPENROUTER_API_KEY,
            base_url="https://openrouter.ai/api/v1"
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str):
        response = self.client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str):
        return self.generate(prompt)

    def get_model_name(self):
        return "openrouter-gpt"

In [12]:
judge_model = OpenRouterLLM()

In [13]:
metrics = [
    AnswerRelevancyMetric(model=judge_model),
    FaithfulnessMetric(model=judge_model),
    ContextualRecallMetric(model=judge_model),
    ContextualPrecisionMetric(model=judge_model)
]

In [14]:
MODEL_E5_LARGE = SentenceTransformer('intfloat/multilingual-e5-large')
COLLECTION_E5_LARGE = "ucu_documents_e5_large"

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2425.87it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<hr>

In [15]:
GEMINI_ANSWER_MODEL = "google/gemini-3-flash-preview"

In [16]:
evaluate(test_cases=create_test_cases(MODEL_E5_LARGE, COLLECTION_E5_LARGE, GEMINI_ANSWER_MODEL), metrics=metrics)

Запит: 'На які категорії поділяються критерії відбору нових працівників в УКУ та яким документом вони регулюються?'
Запит: 'Яка процедура передбачена для чинного працівника УКУ, який бажає взяти участь у конкурсі на вакантну посаду всередині університету?'
Запит: 'Чому відзнаку «Викладач року» в УКУ не можна вважати звичайним професійним конкурсом і які обмеження встановлені щодо повторного отримання нагороди?'
Запит: 'Скільки осіб можуть стати лауреатами відзнаки «Викладач року» протягом одного року та чи можна отримати цю нагороду повторно?'
Запит: 'Опиши хронологію та ключові етапи процесу стратегічного планування цілей в УКУ.'
Запит: 'Яка процедура призначення керівника бакалаврської програми в УКУ згідно з положенням?'
Запит: 'Хто здійснює загальне керівництво магістерською підготовкою в УКУ та кому безпосередньо підпорядковується керівник магістерської програми?'
Запит: 'Хто очолює Бізнес школу УКУ згідно з управлінською структурою?'
Запит: 'За який термін працівника мають повідо

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using openrouter-gpt, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 1.00 because the response directly addresses the input question without any irrelevant statements., error: None)
  - ✅ Faithfulness (score: 0.7777777777777778, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 0.78 because the actual output introduces criteria such as 'професійні та особистісні якості' and a specific evaluation process 'Християнська шкала вартостей', which are not supported by the retrieval context that focuses on 'поведінкові чесноти та компетенції'., error: None)
  - ✅ Contextual Recall (score: 1.0, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 1.00 because all elements of the expected output are directly supported by the nodes in retrieval context, specifically the alignment of behavioral virtues and competencies with the first node and the hiring proce

⚠ WARNING: No hyperparameters logged.
» ]8;id=932002;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 732.18s | token cost: None)
» Test Results (20 total tests):
   » Pass Rate: 85.0% | Passed: 17 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the response directly addresses the input question without any irrelevant statements.', strict_mode=False, evaluation_model='openrouter-gpt', error=None, evaluation_cost=None, verbose_logs='Statements:\n[\n    "Критерії відбору нових працівників в УКУ мають особливості.",\n    "Основними критеріями є поведінкові чесноти та компетенції.",\n    "Поведінкові чесноти та компетенції базуються на місії та цінностях Університету.",\n    "Професійні та особистісні якості визначаються згідно з вимогами конкретної вакансії.",\n    "Християнська шкала вартостей оцінює кваліфікацію, професійні уміння і результати роботи.",\n    "Положення про підбір працівників регулює критерії відбору.",\n    "Додаток 1 до Положення містить перелік чеснот та компетенцій.",\n    "Правила внутрішнього трудового розпоря

<hr>

In [17]:
CLAUDE_ANSWER_MODEL = "anthropic/claude-opus-4.6"

In [18]:
evaluate(test_cases=create_test_cases(MODEL_E5_LARGE, COLLECTION_E5_LARGE, CLAUDE_ANSWER_MODEL), metrics=metrics)

Запит: 'На які категорії поділяються критерії відбору нових працівників в УКУ та яким документом вони регулюються?'
Запит: 'Яка процедура передбачена для чинного працівника УКУ, який бажає взяти участь у конкурсі на вакантну посаду всередині університету?'
Запит: 'Чому відзнаку «Викладач року» в УКУ не можна вважати звичайним професійним конкурсом і які обмеження встановлені щодо повторного отримання нагороди?'
Запит: 'Скільки осіб можуть стати лауреатами відзнаки «Викладач року» протягом одного року та чи можна отримати цю нагороду повторно?'
Запит: 'Опиши хронологію та ключові етапи процесу стратегічного планування цілей в УКУ.'
Запит: 'Яка процедура призначення керівника бакалаврської програми в УКУ згідно з положенням?'
Запит: 'Хто здійснює загальне керівництво магістерською підготовкою в УКУ та кому безпосередньо підпорядковується керівник магістерської програми?'
Запит: 'Хто очолює Бізнес школу УКУ згідно з управлінською структурою?'
Запит: 'За який термін працівника мають повідо

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using openrouter-gpt, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 1.00 because the response directly addresses the input question without any irrelevant statements., error: None)
  - ✅ Faithfulness (score: 0.9230769230769231, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 0.92 because the actual output fails to mention the importance of professional and personal qualities in the selection criteria for new employees at УКУ, which is a key aspect highlighted in the retrieval context., error: None)
  - ✅ Contextual Recall (score: 1.0, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 1.00 because the expected output is fully supported by the relevant nodes in the retrieval context, specifically the alignment of behavioral virtues and competencies mentioned in the first node and the hiring procedure for academic staff detailed in the second n

⚠ WARNING: No hyperparameters logged.
» ]8;id=263549;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 828.56s | token cost: None)
» Test Results (20 total tests):
   » Pass Rate: 90.0% | Passed: 18 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the response directly addresses the input question without any irrelevant statements.', strict_mode=False, evaluation_model='openrouter-gpt', error=None, evaluation_cost=None, verbose_logs='Statements:\n[\n    "Основні критерії відбору нових працівників поділяються на дві категорії.",\n    "Перша категорія - поведінкові чесноти та компетенції.",\n    "Поведінкові чесноти та компетенції базуються на місії та цінностях Університету.",\n    "Детальний перелік поведінкових чеснот та компетенцій наведено у Додатку 1 до відповідного Положення.",\n    "Друга категорія - професійні та особистісні якості.",\n    "Професійні та особистісні якості відповідають конкретним вимогам вакансії.",\n    "Критерії відбору визначаються Положенням про підбір працівників Українського католицького університету.",

<hr>

In [19]:
GPT_ANSWER_MODEL = "openai/gpt-5.4"

In [20]:
evaluate(test_cases=create_test_cases(MODEL_E5_LARGE, COLLECTION_E5_LARGE, GPT_ANSWER_MODEL), metrics=metrics)

Запит: 'На які категорії поділяються критерії відбору нових працівників в УКУ та яким документом вони регулюються?'
Запит: 'Яка процедура передбачена для чинного працівника УКУ, який бажає взяти участь у конкурсі на вакантну посаду всередині університету?'
Запит: 'Чому відзнаку «Викладач року» в УКУ не можна вважати звичайним професійним конкурсом і які обмеження встановлені щодо повторного отримання нагороди?'
Запит: 'Скільки осіб можуть стати лауреатами відзнаки «Викладач року» протягом одного року та чи можна отримати цю нагороду повторно?'
Запит: 'Опиши хронологію та ключові етапи процесу стратегічного планування цілей в УКУ.'
Запит: 'Яка процедура призначення керівника бакалаврської програми в УКУ згідно з положенням?'
Запит: 'Хто здійснює загальне керівництво магістерською підготовкою в УКУ та кому безпосередньо підпорядковується керівник магістерської програми?'
Запит: 'Хто очолює Бізнес школу УКУ згідно з управлінською структурою?'
Запит: 'За який термін працівника мають повідо

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using openrouter-gpt, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using openrouter-gpt, strict=False, 
async_mode=True)...

c:\Users\Olena\Downloads\ucu-employee-support-rag\.venv\lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Answer Relevancy (score: 0.8, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 0.80 because the output includes a reference to a specific appendix that does not directly address the categories of selection criteria or the regulating document, which detracts from its relevance., error: None)
  - ✅ Faithfulness (score: 1.0, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 1.00 because there are no contradictions, indicating that the actual output perfectly aligns with the retrieval context. Great job maintaining consistency!, error: None)
  - ✅ Contextual Recall (score: 1.0, threshold: 0.5, strict: False, evaluation model: openrouter-gpt, reason: The score is 1.00 because the expected output perfectly aligns with the information provided in the nodes in retrieval context, specifically referencing behavioral virtues and competencies as well as the hiring procedure for academic staff., error:

⚠ WARNING: No hyperparameters logged.
» ]8;id=658738;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 673.96s | token cost: None)
» Test Results (20 total tests):
   » Pass Rate: 75.0% | Passed: 15 | Failed: 5

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=0.8, reason='The score is 0.80 because the output includes a reference to a specific appendix that does not directly address the categories of selection criteria or the regulating document, which detracts from its relevance.', strict_mode=False, evaluation_model='openrouter-gpt', error=None, evaluation_cost=None, verbose_logs='Statements:\n[\n    "Критерії відбору нових працівників в УКУ поділяються на такі категорії.",\n    "Поведінкові чесноти та компетенції базуються на місії та цінностях Університету.",\n    "Професійні та особистісні якості відповідають вимогам конкретної вакансії.",\n    "Ці критерії регулюються документом «Положення про підбір працівників Українського католицького університету».",\n    "Поведінкові чесноти та компетенції визначені в Додатку 1."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n    